# Attentional blink: analysis

Raymond, Shapiro & Arnell (1992), *JEP:HPP* 18(3), 849-860, **Experiment 2** (pp. 853-854).

The paper's measure is the percentage of trials on which the probe X was detected, by the
probe's relative serial position (0 = the target, +1..+8 after it), in the **experimental**
condition (identify the white target, then detect the X; only trials with the target
identified correctly) and the **control** condition (ignore the target, detect the X).
Also: false-alarm rates, and the relative serial position of the letter reported as the
target (Figure 3B).

Point `ANALYSIS_RUN` at one run folder, or several separated by `:` (`;` on Windows), or
leave it unset to read every run under `data/` next to this notebook's project. A
simulated run tests this notebook, not the paper: the simulated subject has no blink by
design (`src/raymond1992/simulate.py`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from raymond1992 import analysis

env = os.environ.get("ANALYSIS_RUN")
runs = [Path(p) for p in env.split(os.pathsep)] if env else [Path("..") / "data"]
trials = analysis.load_trials(runs)
df = analysis.clean(trials)
print(f"{len(trials)} rows from {trials['source'].nunique()} run(s); "
      f"{len(df)} completed main trials ({trials['subject'].nunique()} subject(s))")

## The timing these runs used

Whole frames of the measured refresh rate (the paper: 15 ms letters, 75 ms ISI, SOA 90 ms).

In [ ]:
analysis.timing_summary(trials)

## Trial counts per condition and probe position (main blocks)

In [ ]:
df.assign(probe=df['probe_pos'].fillna(-1).astype(int).replace(-1, 'absent')) \
  .groupby(['condition', 'probe']).size().unstack(fill_value=0)

## Probe detection by serial position (Figure 3A)

Experimental: only trials with the target identified correctly.

In [ ]:
by_subject = analysis.detection_by_position(df)
curve = analysis.group_curve(by_subject)
curve.pivot(index='probe_pos', columns='condition', values='mean').round(3)

In [ ]:
soa = float(trials['soa_ms'].median())
fig, ax = plt.subplots(figsize=(6, 4))
for cond, marker, fill in (('control', 'o', 'k'), ('experimental', 'o', 'white')):
    c = curve[curve['condition'] == cond]
    ax.errorbar(c['probe_pos'], 100 * c['mean'], yerr=100 * c['sem'].fillna(0), marker=marker,
                mfc=fill, mec='k', color='k', capsize=2, label=cond)
ax.set_xlabel(f'relative serial position of probe (x {soa:.1f} ms)')
ax.set_ylabel('% probe detected')
ax.set_ylim(0, 100)
ax.set_xticks(range(9))
ax.legend(frameon=False)
plt.show()

## False alarms (an X reported when none was shown)

In [ ]:
analysis.false_alarms(df).groupby('condition')['false_alarm_rate'].mean().round(3)

## Target identification (experimental condition)

In [ ]:
acc = analysis.target_accuracy(df)
print(f"target identified correctly on {100 * acc['p_target_correct'].mean():.1f}% of trials")
analysis.target_report_positions(df).round(3)

## What the paper reported (Experiment 2, 10 subjects; pp. 853-854)

For comparison only. A single run, or a simulated one, is not a replication.

In [ ]:
pd.Series(analysis.PAPER, dtype=object)